# Agent & Tools — Testing Notebook

Interactive tests for:
1. the **data functions** (`src/data.py`) — pure pandas queries, no API calls, and
2. the **analyst agent** (`src/agents/analyst.py`) — the LangGraph tool-calling agent (makes OpenAI calls).

Run top-to-bottom, or tweak the questions and re-run individual cells.

## Setup

In [1]:
import sys
from pathlib import Path

# Make the project root importable so `import src...` works from the notebooks/ folder.
sys.path.insert(0, str(Path.cwd().parent))

## Part 1 — Data functions (no API calls)

These are the plain functions in `src/data.py`. `src/tools.py` wraps them for the
LLM, but here we call them directly so we can test the logic in isolation.

In [2]:
from src import data

data.list_properties()

['Building 120', 'Building 140', 'Building 160', 'Building 17', 'Building 180']

In [3]:
data.list_tenants()

['Tenant 1',
 'Tenant 10',
 'Tenant 11',
 'Tenant 12',
 'Tenant 13',
 'Tenant 14',
 'Tenant 15',
 'Tenant 16',
 'Tenant 17',
 'Tenant 18',
 'Tenant 2',
 'Tenant 3',
 'Tenant 4',
 'Tenant 5',
 'Tenant 6',
 'Tenant 7',
 'Tenant 8',
 'Tenant 9']

### `calculate_pnl` — net P&L with optional filters

In [4]:
# Total P&L across the whole portfolio
data.calculate_pnl()

{'net_pnl': 1533331.87, 'records': 3924}

In [5]:
# P&L for a single property (name matching is case-insensitive)
data.calculate_pnl(property_name="building 17")

{'net_pnl': 352566.81, 'records': 1450}

In [6]:
# P&L for one quarter, revenue only
data.calculate_pnl(quarter="2024-Q1", ledger_type="revenue")

{'net_pnl': 541122.0, 'records': 300}

In [7]:
# Error handling: a property that isn't in the dataset
data.calculate_pnl(property_name="123 Main St")

{'error': "Property '123 Main St' not found."}

### `top_tenants` — ranked by net profit

In [8]:
data.top_tenants(limit=5)

[{'tenant': 'Tenant 7', 'net_pnl': 880512.18},
 {'tenant': 'Tenant 14', 'net_pnl': 391490.29},
 {'tenant': 'Tenant 11', 'net_pnl': 292531.0},
 {'tenant': 'Tenant 13', 'net_pnl': 274344.48},
 {'tenant': 'Tenant 3', 'net_pnl': 204788.42}]

### `breakdown_by` — group net profit by any column

In [9]:
data.breakdown_by(dimension="ledger_group")

[{'ledger_group': 'general_expenses', 'net_pnl': -782154.21},
 {'ledger_group': 'management_fees', 'net_pnl': -471587.49},
 {'ledger_group': 'sales_discounts', 'net_pnl': -185101.75},
 {'ledger_group': 'taxes_and_insurances', 'net_pnl': -100579.32},
 {'ledger_group': 'rental_income', 'net_pnl': 3072754.64}]

In [10]:
# Biggest expense categories for one property
data.breakdown_by(dimension="ledger_category", property_name="Building 17")

[{'ledger_category': 'insurance_in_general', 'net_pnl': -4964.7},
 {'ledger_category': 'rent_discount_taxed', 'net_pnl': -4094.48},
 {'ledger_category': 'other_general_expenses', 'net_pnl': -700.0},
 {'ledger_category': 'proceeds_parking_taxed', 'net_pnl': 16847.4},
 {'ledger_category': 'revenue_rent_taxed', 'net_pnl': 345478.59}]

In [11]:
# Guard rail: an invalid dimension returns an error, not a crash
data.breakdown_by(dimension="not_a_column")

[{'error': "Invalid dimension. Choose from: ['ledger_category', 'ledger_group', 'ledger_type', 'month', 'property_name', 'quarter', 'tenant_name', 'year']"}]

### The tools as the agent sees them

`src/tools.py` exposes these same functions as LangChain tools.

In [3]:
from src.tools import TOOLS

[t.name for t in TOOLS]

['list_properties',
 'list_tenants',
 'calculate_pnl',
 'top_tenants',
 'breakdown_by']

## Part 2 — The analyst agent (makes OpenAI calls)

`ask(question)` runs the full LangGraph loop: the model reads the question,
decides which tool(s) to call, then writes a grounded answer.

In [4]:
from src.agents import ask

### Straightforward questions

In [14]:
print(ask("What is the total P&L for all my properties in 2024?"))

The total P&L for all your properties in 2024 is $1,171,521.55, based on 3,181 records.


In [15]:
print(ask("Who are my top 3 tenants?"))

Your top 3 tenants are:

1. **Tenant 7** - Net P&L: $880,512.18
2. **Tenant 14** - Net P&L: $391,490.29
3. **Tenant 11** - Net P&L: $292,531.00


In [16]:
print(ask("What were my biggest expense categories in 2024?"))

The biggest expense categories in 2024 were:

1. **Interest Mortgage**: -$537,260.17
2. **Success Fees**: -$180,000.00
3. **Rent Discount Taxed**: -$131,250.93
4. **Asset Management Fees**: -$117,000.00
5. **Property Management Fees**: -$96,764.34

These categories represent the largest losses in your financial ledger for the year.


### Compound / comparison questions

In [17]:
print(ask("How does Q1 2025 compare to Q1 2024 in terms of profit?"))

In Q1 2025, the net profit was **$361,810.32** from 743 records, while in Q1 2024, the net profit was **$262,309.07** from 472 records. 

This shows an increase in profit of **$99,501.25** when comparing Q1 2025 to Q1 2024.


### Robustness — vague, out-of-scope, and unknown inputs

In [18]:
# Out of scope: the ledger has no market prices -> should decline, not invent
print(ask("What is the price of my asset at 123 Main St?"))

I currently don't have access to property prices. You may want to check with a real estate listing service or your property management system for the current price of the asset at 123 Main St.


In [19]:
# Unknown property -> should say it's not found
print(ask("What is the P&L for Building 999?"))

The property "Building 999" was not found in the portfolio. Please check the name or provide a different property.


In [20]:
# Vague -> should ask a clarifying question or make a reasonable, stated assumption
print(ask("How am I doing?"))

Could you please clarify what specific information you're looking for? Are you interested in overall profit and loss, performance by property or tenant, or something else?


In [6]:
print(ask("How does this quarter compare to the same period last year? across all the properties and tenatns"))

It appears that there are no records for the specified quarters (Q4 of 2023 and Q4 of 2022) across all properties and tenants. This could mean that there were no transactions recorded during those periods. If you have specific properties or tenants in mind, please let me know!


## Notes

- Data functions are deterministic and unit-testable on their own (Part 1) — the LLM
  is only responsible for *language*, never the arithmetic.
- The agent stays grounded: every figure it reports comes from a tool call.